# Figure 3: prepare the derived analysis datasets

**Input:** `data/Raw/mart_solar_exports_full_release_monthly.csv`.

Select Panels, Cells and Wafers, country/economy records, and January 2022–December 2025. This matches Figure 2's three-product scope. One cleaned row represents one destination, reporting month and product type.

This notebook follows five steps: read and filter, summarize, select the robustness sample, compare peak months, and save. Each step uses ordinary pandas operations.


## 1. Read and filter the raw input


In [2]:
import pandas as pd
from pathlib import Path

root = Path(".")
if not (root / "data").exists():
    root = Path("..")
df = pd.read_csv(root / "data/Raw/mart_solar_exports_full_release_monthly.csv", parse_dates=["Date"])
processed = root / "data/Processed"
processed.mkdir(parents=True, exist_ok=True)

df = df[df["Commodity type"].isin(["Panels", "Cells", "Wafers"])]
df = df[df["Area type"] == "Country or economy"]
df = df[df["Date"].dt.year.between(2022, 2025)].copy()
df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month
if df.duplicated(["Area", "Date", "Commodity type"]).any():
    raise ValueError("Duplicate destination-month-product records need checking.")
if df[["Area", "Region", "Date", "Commodity type", "Capacity (MW)"]].isna().any().any():
    raise ValueError("Missing analysis fields need checking before aggregation.")
if (df["Capacity (MW)"] < 0).any():
    raise ValueError("Negative capacity values need checking.")
df["Capacity (GW)"] = df["Capacity (MW)"] / 1000

## 2. Regional monthly and annual summaries

In [3]:
capacity = ["Capacity (MW)", "Capacity (GW)"]
monthly_product = df.groupby(["year", "month", "Region", "Commodity type"], as_index=False)[capacity].sum()
monthly = df.groupby(["year", "month", "Region"], as_index=False)[capacity].sum()
annual = df.groupby(["Region", "year"])["Capacity (GW)"].sum().unstack()
summary = annual.copy()
summary["Growth 2022–2025 (%)"] = (annual[2025] / annual[2022] - 1) * 100
summary.columns = ["2022 (GW)", "2023 (GW)", "2024 (GW)", "2025 (GW)", "Growth 2022–2025 (%)"]
summary = summary.reset_index().sort_values("2025 (GW)", ascending=False)

## 3. Complete-coverage comparison


In [5]:
month_count = df.groupby(["Area", "Commodity type"])["Date"].transform("nunique")
complete = df[month_count == 48].copy()
complete_monthly = complete.groupby(["year", "month", "Region"], as_index=False)[capacity].sum()
complete_annual = complete.groupby(["Region", "year"])["Capacity (GW)"].sum().unstack()

robustness = pd.DataFrame({
    "All records 2025 (GW)": annual[2025],
    "Complete coverage 2025 (GW)": complete_annual[2025],
    "2025 capacity retained (%)": complete_annual[2025] / annual[2025] * 100,
    "All records growth 2022–2025 (%)": (annual[2025] / annual[2022] - 1) * 100,
    "Complete coverage growth 2022–2025 (%)": (complete_annual[2025] / complete_annual[2022] - 1) * 100,
}).reset_index()
robustness = robustness.sort_values("All records 2025 (GW)", ascending=False)

## 4. Peak-month comparison


In [6]:
all_index = monthly.groupby(["year", "Region"])["Capacity (GW)"].idxmax()
complete_index = complete_monthly.groupby(["year", "Region"])["Capacity (GW)"].idxmax()
all_peaks = monthly.loc[all_index, ["year", "Region", "month"]]
complete_peaks = complete_monthly.loc[complete_index, ["year", "Region", "month"]]
peak_comparison = all_peaks.merge(complete_peaks, on=["year", "Region"], how="left")
peak_comparison.columns = ["year", "Region", "all_peak_month", "complete_peak_month"]
peak_comparison["peak_changed"] = peak_comparison["all_peak_month"] != peak_comparison["complete_peak_month"]

## 5. Save to data/Processed


In [7]:
df.to_csv(processed / "solar_pv_processed_2022_2025.csv", index=False)
monthly_product.to_csv(processed / "figure_3_monthly_region_product_capacity.csv", index=False)
monthly.to_csv(processed / "figure_3_monthly_region_capacity.csv", index=False)
summary.to_csv(processed / "figure_3_annual_region_capacity_gw.csv", index=False)
complete.to_csv(processed / "figure_3_complete_coverage_solar_pv.csv", index=False)
complete_monthly.to_csv(processed / "figure_3_complete_coverage_monthly_capacity.csv", index=False)
robustness.to_csv(processed / "figure_3_robustness_complete_coverage.csv", index=False)
peak_comparison.to_csv(processed / "figure_3_peak_month_comparison.csv", index=False)
print(len(df), "rows saved. Complete-coverage sample:", len(complete), "rows.")
print("Results saved to", processed)

15924 rows saved. Complete-coverage sample: 8064 rows.
Results saved to ../data/Processed
